# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2 Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library, referencing all entities using their `@id` fields for traceability.

### Dataset Source
The dataset is accessible via a Croissant schema URL and describes an experimental mass spectrometry study of human extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded:")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s using the Croissant specification.

**Note**: All entities are referenced by their `@id` field for clarity and reproducibility.

Let's list all record sets and their fields.

In [ ]:
# Gather and print all available record sets and their fields by their @id
record_sets = list(dataset.record_sets())

print(f"Found {len(record_sets)} record sets in this dataset:\n")
rs_dict = {}
for rs in record_sets:
    print(f"Record Set: {rs['@id']} (name: {rs.get('name', None)})")
    fields = rs.get('field', [])
    # Always a list per Croissant
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields and their @id's:")
    for f in fields:
        if isinstance(f, dict):
            print(f"   - {f['@id']} (name: {f.get('name', None)})")
        else:
            print(f"   - {f}")  # Usually it's just an @id string
    rs_dict[rs['@id']] = fields
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

**Note:** Update the following cell to select a record set `@id` and its fields for extraction.

In [ ]:
# Example: Extract all record sets into DataFrames
import collections

record_set_ids = [rs['@id'] for rs in record_sets]
# For demonstration, we'll extract the first record set
main_record_set_id = record_set_ids[0]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Handle empty or non-table record sets gracefully
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame created for {record_set_id}: {len(records)} records, {len(dataframes[record_set_id].columns)} columns")
    else:
        print(f"No tabular records found for record set {record_set_id}")
# For further exploration, use the main record set
print("\nFields/columns for the main record set:")
display(dataframes[main_record_set_id].columns.tolist())
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Process the extracted DataFrame: filter based on a numeric field, normalize, and group.

**Instructions:**
- Pick a numeric field by `@id` (see previous cell's headers).
- Choose a field by `@id` to group by (e.g., a categorical field, if present).

In [ ]:
# Demonstrate EDA: Filtering, normalizing and grouping
df = dataframes[main_record_set_id]

# Select a numeric field to filter and normalize (replace below if necessary)
# Here, we use field @id from schema (manually update if you prefer another):
candidate_numeric_fields = [c for c in df.columns if df[c].dtype.kind in 'iufc']
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]  # e.g., 'cr:MW_kDa' or similar
else:
    raise ValueError("No numeric field found in this record set.")

print(f"Using numeric field: {numeric_field_id}")

# Filter records (example: threshold = 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group field (string/categorical)
candidate_group_fields = [c for c in df.columns if df[c].dtype == 'object']
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id} (showing mean values for numeric columns):")
    display(grouped_df.head())
else:
    print("No suitable group field (categorical) found in this record set.")

## 5. Visualization
Visualize distributions and relationships between fields using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Box plot by group if available
if candidate_group_fields:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This exploratory notebook loaded the FAIR^2 Croissant dataset using the `mlcroissant` API, navigated its record sets, and selected fields for processing by their `@id`. We demonstrated filtering, normalization, grouping, and visualized numeric distributions.

For deeper insights, explore the record sets and fields using their `@id`s as mapped by this Croissant schema.